# Kaggle 09. Direct Transformers + DSPy Prompt Optimization

Этот notebook обходит vLLM полностью. Модель запускается напрямую через `transformers`, поэтому не используется vLLM Triton attention backend, который падает на Tesla T4 для Gemma4.

Цель: проверить zero-shot, deterministic labeled few-shot и, при необходимости, MIPROv2 на небольшом dev-сэмпле. Финальная оценка делается на `MAX_TEST_ROWS=100` примерах test split.


In [ ]:
%pip install -q -U "transformers>=4.56.0" accelerate bitsandbytes "dspy>=3.0.0" kagglehub pandas numpy scikit-learn matplotlib seaborn tqdm

In [ ]:
import json
import logging
import os
import random
from collections import Counter
from pathlib import Path
from types import SimpleNamespace
from typing import Literal

import dspy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("kaggle09_transformers_dspy")
sns.set_theme(style="whitegrid")

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────

KAGGLE_MODEL_HANDLE = os.getenv(
    "KAGGLE_MODEL_HANDLE",
    "google/gemma-4/transformers/gemma-4-e4b-it/1",
)
HF_FALLBACK_MODEL = os.getenv("HF_FALLBACK_MODEL", "google/gemma-4-E4B-it")
KAGGLE_MODEL_LOCAL_PATH = os.getenv("KAGGLE_MODEL_LOCAL_PATH")

OUTPUT_ROOT = Path("/kaggle/working/phase3_transformers_dspy")
TEXT_COLUMN = "text_ru"
SEED = 42

MAX_INPUT_LENGTH = 1536
LM_MAX_TOKENS = 64
LM_TEMPERATURE = 0.0

OPT_TRAIN_PER_CLASS = 6
OPT_DEV_PER_CLASS = 8
MAX_TEST_ROWS = 100
LABELED_FEWSHOT_K = 9

# Direct Transformers is slower than vLLM. Start with LabeledFewShot; enable MIPRO only after smoke works.
RUN_MIPRO = False
MIPRO_AUTO = "light"
MIPRO_MAX_BOOTSTRAPPED = 0
MIPRO_MAX_LABELED = 9
MIPRO_MINIBATCH_SIZE = 24
NUM_THREADS = 1

# "eager" is slower but avoids several optimized-attention backend issues on T4.
ATTN_IMPLEMENTATION = os.getenv("ATTN_IMPLEMENTATION", "eager")

{
    "kaggle_model_handle": KAGGLE_MODEL_HANDLE,
    "hf_fallback_model": HF_FALLBACK_MODEL,
    "text_column": TEXT_COLUMN,
    "run_mipro": RUN_MIPRO,
    "attn_implementation": ATTN_IMPLEMENTATION,
}

In [ ]:
# ── Labels and IO helpers ────────────────────────────────────────────────────

LABELS = [
    "none",
    "appeal to authority",
    "appeal to majority",
    "appeal to nature",
    "appeal to tradition",
    "appeal to worse problems",
    "false dilemma",
    "hasty generalization",
    "slippery slope",
]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
ACCEPTED_STATUSES = {"ok", "repaired_ok"}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_label(value) -> str:
    return " ".join(str(value).strip().lower().replace("_", " ").replace("-", " ").split())


_NORM = {normalize_label(label): label for label in LABELS}


def canonical_label(value) -> str | None:
    if value is None:
        return None
    n = normalize_label(value)
    if n in _NORM:
        return _NORM[n]
    for label in LABELS:
        if normalize_label(label) in n:
            return label
    return None


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")


set_seed(SEED)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Resolve dataset and model ────────────────────────────────────────────────

INPUT_JSONL_CANDIDATES = sorted(Path("/kaggle/input").rglob("*.jsonl"))
assert INPUT_JSONL_CANDIDATES, "Upload cocolofa_ru_v2*.jsonl to /kaggle/input first."

DATASET_PATH = (
    next((p for p in INPUT_JSONL_CANDIDATES if p.name == "cocolofa_ru_v2_explanations.jsonl"), None)
    or next((p for p in INPUT_JSONL_CANDIDATES if p.name == "cocolofa_ru_v2.jsonl"), None)
    or next((p for p in INPUT_JSONL_CANDIDATES if p.name == "cocolofa_ru_v2_masked.jsonl"), None)
)
assert DATASET_PATH is not None, f"No supported dataset found: {[p.name for p in INPUT_JSONL_CANDIDATES]}"
print("Dataset:", DATASET_PATH)


def has_model_files(path: Path) -> bool:
    return (
        (path / "config.json").exists()
        and (any(path.glob("*.safetensors")) or any(path.glob("*.bin")) or (path / "model.safetensors.index.json").exists())
    )


def find_local_model_dir() -> str | None:
    if KAGGLE_MODEL_LOCAL_PATH and has_model_files(Path(KAGGLE_MODEL_LOCAL_PATH)):
        return KAGGLE_MODEL_LOCAL_PATH

    hints = [part for part in KAGGLE_MODEL_HANDLE.lower().replace("/", "-").split("-") if part]
    important = [h for h in hints if h in {"gemma", "4", "e4b", "26b", "a4b", "qwen", "llama", "mistral"}]
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if not root.exists():
            continue
        for config_path in root.rglob("config.json"):
            parent = config_path.parent
            text = str(parent).lower()
            if has_model_files(parent) and (not important or all(h in text for h in important[:3])):
                return str(parent)
    return None


def resolve_model_path() -> str:
    local_dir = find_local_model_dir()
    if local_dir:
        logger.info("Using local model directory: %s", local_dir)
        return local_dir

    try:
        import kagglehub

        logger.info("Downloading Kaggle model via kagglehub: %s", KAGGLE_MODEL_HANDLE)
        return kagglehub.model_download(KAGGLE_MODEL_HANDLE)
    except Exception as exc:
        logger.warning("KaggleHub download failed: %r", exc)
        logger.warning("Falling back to Hugging Face: %s", HF_FALLBACK_MODEL)
        return HF_FALLBACK_MODEL


MODEL_PATH = resolve_model_path()
print("Model path:", MODEL_PATH)

In [ ]:
# ── Load Gemma/Qwen/etc through direct Transformers ──────────────────────────

def load_model_and_tokenizer(model_path: str):
    processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
    tokenizer = getattr(processor, "tokenizer", processor)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs = {
        "trust_remote_code": True,
        "device_map": "auto",
    }
    if ATTN_IMPLEMENTATION:
        kwargs["attn_implementation"] = ATTN_IMPLEMENTATION
    if torch.cuda.is_available():
        kwargs.update(
            {
                "torch_dtype": torch.float16,
                "quantization_config": BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_use_double_quant=True,
                ),
            }
        )

    model = AutoModelForCausalLM.from_pretrained(model_path, **kwargs)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return model, tokenizer


def model_device(model) -> torch.device:
    return next(model.parameters()).device


model, tokenizer = load_model_and_tokenizer(MODEL_PATH)
print("Loaded model on:", model_device(model))
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# ── Dataset preparation ─────────────────────────────────────────────────────

raw = load_jsonl(DATASET_PATH)
df = pd.DataFrame(raw)
assert not df.empty and "split" in df.columns

if TEXT_COLUMN not in df.columns:
    TEXT_COLUMN = "text_masked" if "text_masked" in df.columns else "text"
    logger.warning("Fallback text column: %s", TEXT_COLUMN)

if "translation_status" in df.columns:
    df = df[df["translation_status"].isin(ACCEPTED_STATUSES)].copy()


def infer_label(row: pd.Series) -> str:
    for col in ("label_str", "label"):
        if col in row and pd.notna(row[col]):
            return str(row[col])
    if "label_id" in row and pd.notna(row["label_id"]):
        return ID_TO_LABEL[int(row["label_id"])]
    raise ValueError(row.to_dict())


df["label_str"] = df.apply(infer_label, axis=1).map(lambda v: canonical_label(v) or v)
assert not (bad := sorted(set(df["label_str"]) - set(LABELS))), bad

df = df[df["split"].isin(["train", "dev", "test"])].copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
df = df[df[TEXT_COLUMN].str.strip().ne("")].copy()

train_df = df[df["split"] == "train"].reset_index(drop=True)
dev_df = df[df["split"] == "dev"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

display(pd.DataFrame({"split": ["train", "dev", "test"], "rows": [len(train_df), len(dev_df), len(test_df)]}))
display(df["label_str"].value_counts().reindex(LABELS).rename("count").to_frame())

In [ ]:
# ── Stratified optimization/evaluation samples ─────────────────────────────

def stratified_sample(src: pd.DataFrame, per_class: int, seed: int) -> pd.DataFrame:
    parts = []
    for label in LABELS:
        group = src[src["label_str"] == label]
        assert len(group), f"No rows for {label!r}"
        parts.append(group.sample(n=min(per_class, len(group)), random_state=seed))
    out = pd.concat(parts, ignore_index=True)
    out["_ord"] = out["label_str"].map(LABEL_TO_ID)
    return out.sort_values("_ord").drop(columns="_ord").reset_index(drop=True)


def round_robin(src: pd.DataFrame) -> pd.DataFrame:
    groups = {label: src[src["label_str"] == label].reset_index(drop=True) for label in LABELS}
    rows = []
    for i in range(max(len(group) for group in groups.values())):
        for label in LABELS:
            if i < len(groups[label]):
                rows.append(groups[label].iloc[i])
    return pd.DataFrame(rows).reset_index(drop=True)


train_opt = round_robin(stratified_sample(train_df, OPT_TRAIN_PER_CLASS, SEED))
dev_opt = stratified_sample(dev_df, OPT_DEV_PER_CLASS, SEED)
test_eval = test_df.sample(n=min(MAX_TEST_ROWS, len(test_df)), random_state=SEED).reset_index(drop=True) if MAX_TEST_ROWS else test_df.copy()

print(f"train_opt={len(train_opt)} dev_opt={len(dev_opt)} test_eval={len(test_eval)}")

In [ ]:
# ── Direct Transformers LM wrapper for DSPy ─────────────────────────────────

class TransformersDSPyLM(dspy.BaseLM):
    def __init__(
        self,
        hf_model,
        tokenizer,
        model_name: str = "local-transformers",
        max_input_length: int = 1536,
        temperature: float = 0.0,
        max_tokens: int = 64,
    ):
        super().__init__(model=model_name, model_type="chat", temperature=temperature, max_tokens=max_tokens, cache=False)
        self.hf_model = hf_model
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length

    def copy(self, **kwargs):
        # Do not deepcopy the loaded HF model. DSPy optimizers may call lm.copy(...).
        new = TransformersDSPyLM(
            self.hf_model,
            self.tokenizer,
            model_name=self.model,
            max_input_length=self.max_input_length,
            temperature=self.kwargs.get("temperature", 0.0),
            max_tokens=self.kwargs.get("max_tokens", 64),
        )
        new.kwargs = {**self.kwargs}
        for key, value in kwargs.items():
            if value is None:
                new.kwargs.pop(key, None)
            else:
                new.kwargs[key] = value
        return new

    def _render_messages(self, messages: list[dict]) -> str:
        if getattr(self.tokenizer, "chat_template", None):
            return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        rendered = []
        for message in messages:
            rendered.append(f"{message.get('role', 'user').upper()}: {message.get('content', '')}")
        rendered.append("ASSISTANT:")
        return "\n\n".join(rendered)

    def forward(self, prompt=None, messages=None, **kwargs):
        messages = messages or [{"role": "user", "content": prompt}]
        prompt_text = self._render_messages(messages)
        merged = {**self.kwargs, **kwargs}
        max_new_tokens = int(merged.get("max_tokens") or merged.get("max_new_tokens") or LM_MAX_TOKENS)
        temperature = float(merged.get("temperature") or 0.0)

        encoded = self.tokenizer(
            [prompt_text],
            truncation=True,
            max_length=self.max_input_length,
            padding=True,
            return_tensors="pt",
        )
        device = model_device(self.hf_model)
        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            generated = self.hf_model.generate(
                **encoded,
                max_new_tokens=max_new_tokens,
                do_sample=temperature > 0,
                temperature=temperature if temperature > 0 else None,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        new_tokens = generated[:, encoded["input_ids"].shape[1]:]
        text = self.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

        message = SimpleNamespace(content=text)
        choice = SimpleNamespace(message=message)
        return SimpleNamespace(model=self.model, choices=[choice], usage={})


lm = TransformersDSPyLM(
    model,
    tokenizer,
    model_name=f"transformers/{Path(str(MODEL_PATH)).name or 'local'}",
    max_input_length=MAX_INPUT_LENGTH,
    temperature=LM_TEMPERATURE,
    max_tokens=LM_MAX_TOKENS,
)
dspy.configure(lm=lm)

print("DSPy:", getattr(dspy, "__version__", "unknown"))
print("Smoke:", lm(messages=[{"role": "user", "content": "Ответь ровно JSON: {\"label\": \"none\"}"}]))

In [ ]:
# ── DSPy signature, examples, metric ────────────────────────────────────────

LabelLiteral = Literal[
    "none",
    "appeal to authority",
    "appeal to majority",
    "appeal to nature",
    "appeal to tradition",
    "appeal to worse problems",
    "false dilemma",
    "hasty generalization",
    "slippery slope",
]


class FallacySignature(dspy.Signature):
    """Классифицируй русский аргумент в один класс логической ошибки.

    Используй только заданную таксономию. Если ошибка не выражена явно, выбирай `none`.
    Оценивай аргументативный переход, а не истинность фактов.
    """

    text_ru: str = dspy.InputField(desc="Русский аргументативный текст.")
    label: LabelLiteral = dspy.OutputField(
        desc="Ровно одна метка из: " + ", ".join(LABELS) + ". Для корректных рассуждений — none."
    )


def make_program() -> dspy.Predict:
    return dspy.Predict(FallacySignature)


def to_examples(src: pd.DataFrame) -> list[dspy.Example]:
    return [
        dspy.Example(text_ru=str(getattr(row, TEXT_COLUMN)), label=str(row.label_str)).with_inputs("text_ru")
        for row in src.itertuples(index=False)
    ]


trainset = to_examples(train_opt)
devset = to_examples(dev_opt)
testset = to_examples(test_eval)

train_counts = train_df["label_str"].value_counts().to_dict()
raw_weights = {label: 1.0 / np.sqrt(max(train_counts.get(label, 1), 1)) for label in LABELS}
max_weight = max(raw_weights.values())
CLASS_WEIGHTS = {label: raw_weights[label] / max_weight for label in LABELS}
CLASS_WEIGHTS["none"] = max(CLASS_WEIGHTS["none"], 0.75)


def weighted_exact_match(example, pred, trace=None) -> float:
    gold = canonical_label(getattr(example, "label", None))
    predicted = canonical_label(getattr(pred, "label", None))
    if gold is None or predicted is None:
        return 0.0
    return float(CLASS_WEIGHTS.get(gold, 1.0)) if gold == predicted else 0.0


pd.DataFrame([{"label": l, "count": train_counts.get(l, 0), "weight": round(CLASS_WEIGHTS[l], 3)} for l in LABELS])

In [ ]:
# ── Evaluation helpers ──────────────────────────────────────────────────────

def compute_metrics(rows: list[dict]) -> dict:
    y_true = [row["gold_label"] for row in rows]
    y_pred = [row["pred_label"] for row in rows]
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    weighted_denom = sum(CLASS_WEIGHTS.get(g, 1.0) for g in y_true)
    weighted_num = sum(CLASS_WEIGHTS.get(g, 1.0) for g, pred in zip(y_true, y_pred) if g == pred)
    none_fp = sum(1 for g, pred in zip(y_true, y_pred) if g == "none" and pred != "none")
    none_n = y_true.count("none")
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(np.mean(f)),
        "weighted_exact_match": float(weighted_num / weighted_denom) if weighted_denom else 0.0,
        "none_fpr": float(none_fp / none_n) if none_n else 0.0,
        "invalid_rate": float(sum(1 for pred in y_pred if pred not in LABELS) / len(y_pred)),
        "per_class": {label: {"p": float(p[i]), "r": float(r[i]), "f1": float(f[i]), "n": int(s[i])} for i, label in enumerate(LABELS)},
    }


def evaluate_program(program, eval_df: pd.DataFrame, name: str, out_dir: Path):
    rows = []
    for row in tqdm(list(eval_df.itertuples(index=False)), desc=f"Eval {name}"):
        text = str(getattr(row, TEXT_COLUMN))
        gold_label = str(row.label_str)
        try:
            pred = program(text_ru=text)
            raw_label = getattr(pred, "label", None)
            error = None
        except Exception as exc:
            raw_label = None
            error = repr(exc)
        pred_label = canonical_label(raw_label) or "__invalid__"
        rows.append(
            {
                "program": name,
                "split": getattr(row, "split"),
                "sample_id": int(getattr(row, "sample_id")) if hasattr(row, "sample_id") else None,
                "gold_label": gold_label,
                "pred_label": pred_label,
                "raw_label": str(raw_label),
                "error": error,
                "text": text,
            }
        )

    metrics = compute_metrics(rows)
    out_dir.mkdir(parents=True, exist_ok=True)
    write_json(out_dir / f"{name}_metrics.json", metrics)
    write_jsonl(out_dir / f"{name}_predictions.jsonl", rows)
    write_json(
        out_dir / f"{name}_report.json",
        classification_report(
            [r["gold_label"] for r in rows],
            [r["pred_label"] for r in rows],
            labels=LABELS,
            output_dict=True,
            zero_division=0,
        ),
    )
    return metrics, rows


def summarize(metrics_dict: dict) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "program": name,
                "accuracy": metrics["accuracy"],
                "macro_f1": metrics["macro_f1"],
                "weighted_em": metrics["weighted_exact_match"],
                "none_fpr": metrics["none_fpr"],
                "invalid_rate": metrics["invalid_rate"],
            }
            for name, metrics in metrics_dict.items()
        ]
    ).sort_values(["weighted_em", "macro_f1"], ascending=False)

In [ ]:
# ── Zero-shot and deterministic LabeledFewShot ─────────────────────────────

programs = {}
dev_metrics = {}

programs["zero_shot"] = make_program()
dev_metrics["zero_shot"], _ = evaluate_program(programs["zero_shot"], dev_opt, "dev_zero_shot", OUTPUT_ROOT / "dev")

programs["labeled_fewshot"] = dspy.LabeledFewShot(k=LABELED_FEWSHOT_K).compile(
    make_program(),
    trainset=trainset,
    sample=False,
)
dev_metrics["labeled_fewshot"], _ = evaluate_program(
    programs["labeled_fewshot"], dev_opt, "dev_labeled_fewshot", OUTPUT_ROOT / "dev"
)

dev_summary = summarize(dev_metrics)
display(dev_summary)

In [ ]:
# ── Optional MIPROv2 ─────────────────────────────────────────────────────────

if RUN_MIPRO:
    mipro = dspy.MIPROv2(
        metric=weighted_exact_match,
        auto=MIPRO_AUTO,
        num_threads=NUM_THREADS,
        max_errors=20,
        seed=SEED,
        verbose=True,
    )

    mipro_program = mipro.compile(
        make_program(),
        trainset=trainset,
        valset=devset,
        max_bootstrapped_demos=MIPRO_MAX_BOOTSTRAPPED,
        max_labeled_demos=MIPRO_MAX_LABELED,
        minibatch=True,
        minibatch_size=min(MIPRO_MINIBATCH_SIZE, len(devset)),
        minibatch_full_eval_steps=2,
        seed=SEED,
    )

    programs["mipro_light"] = mipro_program
    dev_metrics["mipro_light"], _ = evaluate_program(mipro_program, dev_opt, "dev_mipro_light", OUTPUT_ROOT / "dev")
else:
    print("MIPROv2 skipped. Set RUN_MIPRO=True after zero/few-shot smoke is stable.")

dev_summary = summarize(dev_metrics)
display(dev_summary)
write_json(OUTPUT_ROOT / "dev_summary.json", dev_metrics)
dev_summary.to_csv(OUTPUT_ROOT / "dev_summary.csv", index=False)

In [ ]:
# ── Final held-out test evaluation ──────────────────────────────────────────

best_name = dev_summary.iloc[0]["program"]
test_metrics = {}
test_rows_by = {}

for name, program in programs.items():
    metrics, rows = evaluate_program(program, test_eval, f"test_{name}", OUTPUT_ROOT / "test")
    test_metrics[name] = metrics
    test_rows_by[name] = rows

test_summary = summarize(test_metrics)
display(test_summary)
write_json(OUTPUT_ROOT / "test_summary.json", test_metrics)
test_summary.to_csv(OUTPUT_ROOT / "test_summary.csv", index=False)

best_rows = test_rows_by[best_name]
cm = confusion_matrix([r["gold_label"] for r in best_rows], [r["pred_label"] for r in best_rows], labels=LABELS)

plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.title(f"Direct Transformers + DSPy | {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Gold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / f"cm_{best_name}.png", dpi=150)
plt.show()

pc = pd.DataFrame(test_metrics[best_name]["per_class"]).T.reset_index(names="label")
display(pc.sort_values("f1", ascending=False))

In [ ]:
# ── Save compact run config ─────────────────────────────────────────────────

write_json(
    OUTPUT_ROOT / "config.json",
    {
        "dataset_path": str(DATASET_PATH),
        "model_path": str(MODEL_PATH),
        "kaggle_model_handle": KAGGLE_MODEL_HANDLE,
        "hf_fallback_model": HF_FALLBACK_MODEL,
        "text_column": TEXT_COLUMN,
        "seed": SEED,
        "max_input_length": MAX_INPUT_LENGTH,
        "lm_max_tokens": LM_MAX_TOKENS,
        "max_test_rows": MAX_TEST_ROWS,
        "labeled_fewshot_k": LABELED_FEWSHOT_K,
        "run_mipro": RUN_MIPRO,
        "attn_implementation": ATTN_IMPLEMENTATION,
    },
)

print("Saved to:", OUTPUT_ROOT)